In [2]:
# import libraries
import pandas as pd
import sys
import os

sys.path.append('../')
from src.data_loader import load_cement_data

In [3]:
df = load_cement_data()

In [4]:
# add warning filter to ignore warnings
import warnings
warnings.filterwarnings('ignore')

In [5]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [6]:
df.head()

,date,site_id,region,behavior,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity
0,2022-01-01,SITE_001,North,aggressive,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,448
1,2022-01-02,SITE_001,North,aggressive,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,448
2,2022-01-03,SITE_001,North,aggressive,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,448
3,2022-01-04,SITE_001,North,aggressive,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,448
4,2022-01-05,SITE_001,North,aggressive,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,448


In [7]:
site_id = 'SITE_001'
site_df = df[df['site_id'] == site_id].copy()


In [8]:
site_df.set_index('date', inplace=True)
site_df = site_df.sort_index()

In [9]:
site_df.columns

Index(['site_id', 'region', 'behavior', 'cement_type', 'planned_pour_tonnes',
       'consumed_tonnes', 'opening_inventory_tonnes', 'deliveries_tonnes',
       'closing_inventory_tonnes', 'rain_mm', 'avg_temp_c', 'silo_capacity'],
      dtype='object')

In [10]:
site_df.shape


(1096, 12)

In [11]:
site_df.head()

,site_id,region,behavior,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity
date,,,,,,,,,,,,
2022-01-01,SITE_001,North,aggressive,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,448
2022-01-02,SITE_001,North,aggressive,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,448
2022-01-03,SITE_001,North,aggressive,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,448
2022-01-04,SITE_001,North,aggressive,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,448
2022-01-05,SITE_001,North,aggressive,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,448


In [12]:
y = site_df['consumed_tonnes']

exog = site_df[['planned_pour_tonnes', 'rain_mm', 'avg_temp_c']]

In [13]:
split_index = int(len(site_df) * 0.8)
X_train, X_test = exog.iloc[:split_index], exog.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

In [14]:
model = SARIMAX(
  y_train,
  exog = X_train,
  order = (1, 1, 1),
  seasonal_order = (1, 0, 1, 7),
  enforce_stationarity = False,
  enforce_invertibility = False
)

results = model.fit(disp=False)
results.summary()

/home/whizic/.local/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/home/whizic/.local/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


<class 'statsmodels.iolib.summary.Summary'>
"""
                                     SARIMAX Results                                     
=========================================================================================
Dep. Variable:                   consumed_tonnes   No. Observations:                  876
Model:             SARIMAX(1, 1, 1)x(1, 0, 1, 7)   Log Likelihood               -3381.155
Date:                           Fri, 17 Jul 2026   AIC                           6778.310
Time:                                   00:18:19   BIC                           6816.421
Sample:                               01-01-2022   HQIC                          6792.895
                                    - 05-25-2024                                         
Covariance Type:                             opg                                         
=======================================================================================
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
planned_pour_tonnes     0.5504      0.039     14.239      0.000       0.475       0.626
rain_mm                -0.8606      0.067    -12.835      0.000      -0.992      -0.729
avg_temp_c              0.0497      0.052      0.962      0.336      -0.052       0.151
ar.L1                   0.0849      0.034      2.473      0.013       0.018       0.152
ma.L1                  -0.9970      0.004   -225.112      0.000      -1.006      -0.988
ar.S.L7                -0.0430      0.592     -0.073      0.942      -1.202       1.116
ma.S.L7                 0.0188      0.591      0.032      0.975      -1.140       1.177
sigma2                143.4096      8.812     16.275      0.000     126.139     160.681
===================================================================================
Ljung-Box (L1) (Q):                   0.00   Jarque-Bera (JB):                35.97
Prob(Q):                              1.00   Prob(JB):                         0.00
Heteroskedasticity (H):               0.93   Skew:                            -0.43
Prob(H) (two-sided):                  0.55   Kurtosis:                         2.48
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
"""

In [15]:
# SARIMAX Forecast
sarimax_forecast = results.predict(
  start=y_test.index[0], 
  end=y_test.index[-1], 
  exog=X_test
  )

In [16]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=y_test.index,
    y=y_test,
    mode='lines',
    name='Actual',
    line=dict(color='black')
))

fig.add_trace(go.Scatter(
    x=y_test.index,
    y=sarimax_forecast,
    mode='lines',
    name='Forecast',
    line=dict(color='blue')
))

fig.update_layout(
    title=f'SARIMAX Forecast vs Actual - {site_df}',
    xaxis_title='Date',
    yaxis_title='Cement Consumed Tonnes',
    legend=dict(x=0, y=1),
    template='plotly_white',
    width=1000,
    height=450
)

fig.show()

In [17]:
import numpy as np
from sklearn.metrics import mean_squared_error

def print_metrics(y_true, y_pred, label):
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    rmse = np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))
    print(f"{label} - MAPE: {mape:.2f}%, RMSE: {rmse:.2f} tonnes")

In [18]:
from sklearn.ensemble import RandomForestRegressor

In [19]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [20]:
rf_forecast = rf.predict(X_test)

In [21]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=y_test.index,
    y=y_test,
    mode='lines',
    name='Actual',
    line=dict(color='black')
))

fig.add_trace(go.Scatter(
    x=y_test.index,
    y=rf_forecast,
    mode='lines',
    name='Forecast',
    line=dict(color='red')
))

fig.update_layout(
    title=f'Random Forest Forecast vs Actual - {site_df}',
    xaxis_title='Date',
    yaxis_title='Cement Consumed Tonnes',
    legend=dict(x=0, y=1),
    template='plotly_white',
    width=1000,
    height=450
)

fig.show()

In [22]:
# compare metrics for both models
print_metrics(y_test, sarimax_forecast, 'SARIMAX Model')
print_metrics(y_test, rf_forecast, 'Random Forest Model')

SARIMAX Model - MAPE: 36.17%, RMSE: 11.41 tonnes
Random Forest Model - MAPE: 34.86%, RMSE: 11.45 tonnes
